# Mask-Attention Actor-Critic

---
## 目的
Mask-Attentionの仕組みを理解し，ゲームタスクを用いて強化学習を行う．
学習後のエージェントの可視化とエージェントの注視領域を確認し，強化学習によりAIがどのような視点を持っているかの確認を行う．

## Mask-Attention Actor-Critic
Mask-Attention Actor-Critic（Mask AC）は，`actor_critic.ipynb`のActor-CriticにAttention機構を導入した手法です．Mask ACでは，Policy branchとValue branchに対しAttention機構を導入することで，各ブランチの出力値に対するネットワークの注視領域を表現したAttention mapを獲得します．このAttention mapはMask-Attentionと呼ばれます．モデルの推論時において各ブランチのMask-Attentionを可視化することで，方策および状態価値関数に対する判断根拠の視覚的説明を実現しています．

### Attention機構
Mask ACでは，Policy branchとValue branchにAttention機構を導入することで，獲得したMask-Attentionを考慮して方策および状態価値関数を出力します．Attention機構は，各ブランチ内における中間層の特徴マップに対し，Mask-Attentionを用いてマスク処理を行います．特徴マップに対するMask-Attentionを用いた処理は以下の式で表されます．ここで，$s_t$は状態，$F(s_t)$は各ブランチ内における中間層の特徴マップ，$M(s_t)$はMask-Attention，$F'(s_t)$はマスク処理後の特徴マップです．

$$
F'(s_t)=F(s_t)\cdot M(s_t)
$$

### Convolutional LSTM
通常のActor-Criticでは，入力画像に対する時間・空間情報が欠落してしまうため，Mask-Attentionを出力することができません．そのため，特徴抽出部に時空間情報を考慮するLSTMであるConvolutional LSTM（ConvLSTM）を追加します．特徴抽出部は畳み込み層3層とConvLSTMで構成されます．Policy branchとValue branchは，それぞれ1×1の畳み込み層とSigmoid関数からMask-Attentionを獲得し，中間特徴マップに乗算（マスク処理）してから，全結合層で方策・状態価値を出力します．


## 準備
下記のプログラムを実行して，実験に必要な追加ライブラリをインストールする．

In [ ]:
!pip install -q "gymnasium[atari,other]" ale-py "imageio[ffmpeg]"

## モジュールのインポートとGPUの確認
はじめに必要なモジュールをインポートする．

今回はPyTorchに加えて，Pongを実行するためのシミュレータであるGymnasium（gymnasium）をインポートする．
そして，GPUが使用可能かどうかを確認する．

In [ ]:
import time
import datetime
import random
import collections
import cv2
import imageio
from base64 import b64encode

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

import gymnasium as gym
import ale_py

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Use device:', device)

# フレーム画像のリストをmp4動画に変換し，Notebook上で再生する
def frames_to_html5(frames, path, fps=15):
    imageio.mimwrite(path, frames, fps=fps)
    mp4 = open(path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    return HTML(f'<video width="320" height="420" controls><source src="{data_url}" type="video/mp4"></video>')

## シード値の固定

In [ ]:
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms = True

## OpenAI GymによるPong環境の定義
[Gymnasium](https://gymnasium.farama.org/)は，様々な種類の環境を提供しているモジュールです．
今回は，Gymnasiumで利用可能なAtari2600のゲームであるPongを使用します．
Pongの行動数は6ですが，実質的な行動はパドルを上下どちらかに移動させる2種類のみです（詳細は`deep_q_network.ipynb`を参照）．

In [ ]:
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False)

obs, info = env.reset(seed=seed, options=None)
print('observation space:', env.observation_space)
print('action space:', env.action_space)
print('initial observation:', obs.shape)

## 環境の前処理
Atari環境の学習を安定・効率化するため，`deep_q_network.ipynb`と同様の前処理を適用します．

* MaxAndSkipEnv：1ステップ実行毎に，4フレームゲームを進める（skip frame）
* FireResetEnv：エピソード（ゲーム）開始にFireを実行しなければ開始されない環境でのreset関数の設定
* ProcessFrame84：210×160のRGB画像を84×84のグレースケール画像に変換
* ImageToPyTorch：観測情報（画像）のshapeをHWC（高さ，幅，チャネル）からCHW（チャネル，高さ，幅）に変換
* ScaledFloatFrame：画像（0から255）を0.0から1.0の範囲で正規化

Mask-Attentionでは，ConvLSTMによって時系列情報をネットワーク内部で保持するため，`deep_q_network.ipynb`や`actor_critic.ipynb`で用いた`BufferWrapper`（直近4フレームをスタック）は使用せず，1フレームのみを入力とします．

In [ ]:
class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env=None, skip=4):
        super(MaxAndSkipEnv, self).__init__(env)
        self._obs_buffer = collections.deque(maxlen=2)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        done = None
        for _ in range(self._skip):
            obs, reward, done, truncated, info = self.env.step(action)
            self._obs_buffer.append(obs)
            total_reward += reward
            if done:
                break
        max_frame = np.max(np.stack(self._obs_buffer), axis=0)
        return max_frame, total_reward, done, truncated, info

class FireResetEnv(gym.Wrapper):
    def __init__(self, env=None):
        super(FireResetEnv, self).__init__(env)
        assert env.unwrapped.get_action_meanings()[1] == 'FIRE'
        assert len(env.unwrapped.get_action_meanings()) >= 3

    def step(self, action):
        return self.env.step(action)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        obs, _, done, truncated, _ = self.env.step(1)
        if done or truncated:
            obs, info = self.env.reset(**kwargs)
        obs, _, done, truncated, _ = self.env.step(2)
        if done or truncated:
            obs, info = self.env.reset(**kwargs)
        return obs, info

class ProcessFrame84(gym.ObservationWrapper):
    def __init__(self, env=None):
        super(ProcessFrame84, self).__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)

    def observation(self, obs):
        return ProcessFrame84.process(obs)

    @staticmethod
    def process(frame):
        if frame.size == 210 * 160 * 3:
            img = np.reshape(frame, [210, 160, 3]).astype(np.float32)
        elif frame.size == 250 * 160 * 3:
            img = np.reshape(frame, [250, 160, 3]).astype(np.float32)
        else:
            assert False, "Unknown resolution."
        img = img[:, :, 0] * 0.299 + img[:, :, 1] * 0.587 + img[:, :, 2] * 0.114
        resized_screen = cv2.resize(img, (84, 110), interpolation=cv2.INTER_AREA)
        x_t = resized_screen[18:102, :]
        x_t = np.reshape(x_t, [84, 84, 1])
        return x_t.astype(np.uint8)

class ImageToPyTorch(gym.ObservationWrapper):
    def __init__(self, env):
        super(ImageToPyTorch, self).__init__(env)
        old_shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(old_shape[-1], old_shape[0], old_shape[1]),
                                                dtype=np.float32)

    def observation(self, observation):
        return np.moveaxis(observation, 2, 0)

class ScaledFloatFrame(gym.ObservationWrapper):
    def observation(self, obs):
        return np.array(obs).astype(np.float32) / 255.0

### 前処理の適用
環境に対して必要となる前処理を適用します．

In [ ]:
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False)

env = MaxAndSkipEnv(env)
env = FireResetEnv(env)
env = ProcessFrame84(env)
env = ImageToPyTorch(env)
env = ScaledFloatFrame(env)

## Convolutional LSTM
LSTMの内部構造に畳み込み層を導入したConvLSTMを定義します．通常のLSTMが全結合層でゲートを計算するのに対し，ConvLSTMでは畳み込み層でゲートを計算するため，特徴マップの空間構造を保持したまま時系列情報を扱うことができます．

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super(ConvLSTMCell, self).__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size[0] // 2, kernel_size[1] // 2

        # 入力ゲート，忘却ゲート，出力ゲート，セル候補の4つをまとめて1回の畳み込みで計算する
        self.conv = nn.Conv2d(
            in_channels=input_dim + hidden_dim,
            out_channels=4 * hidden_dim,
            kernel_size=kernel_size,
            padding=padding
        )

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)

        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch_size, image_size, device):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device))

## ネットワーク構造
Mask ACのネットワークを定義します．Mask ACのネットワーク構造はFeature extractor，ConvLSTM，Policy branch，Value branchから構成されます．
Feature extractorは畳み込み層3層で構成され，ゲーム画面から空間的な特徴を抽出します．ConvLSTMは，Feature extractorが出力する特徴マップを時系列方向に伝播し，時空間情報を考慮した特徴マップを出力します．
Policy branchとValue branchは，それぞれConvLSTMの出力に対して1×1畳み込み+Sigmoidで空間的なMask-Attentionを計算し，特徴マップに乗算してから，畳み込み層と全結合層を通して方策（行動のスコア）・状態価値を出力します．

In [ ]:
class MaskAC(nn.Module):
    def __init__(self, input_shape, n_actions, hidden_dim=64):
        super(MaskAC, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        self.hidden_dim = hidden_dim
        self.conv_hw = self._get_conv_out_hw(input_shape)
        self.convlstm = ConvLSTMCell(input_dim=64, hidden_dim=hidden_dim, kernel_size=(3, 3))

        conv_out_size = 32 * self.conv_hw[0] * self.conv_hw[1]

        # Policy branch（Mask-Attention + 方策の出力）
        self.policy_att = nn.Conv2d(hidden_dim, 1, kernel_size=1)
        self.policy_conv = nn.Conv2d(hidden_dim, 32, kernel_size=1)
        self.policy_fc = nn.Linear(conv_out_size, n_actions)

        # Value branch（Mask-Attention + 状態価値の出力）
        self.value_att = nn.Conv2d(hidden_dim, 1, kernel_size=1)
        self.value_conv = nn.Conv2d(hidden_dim, 32, kernel_size=1)
        self.value_fc = nn.Linear(conv_out_size, 1)

    def _get_conv_out_hw(self, shape):
        o = self.conv(torch.zeros(1, *shape))
        return o.size(2), o.size(3)

    def forward(self, x, lstm_state):
        feature = self.conv(x)
        h, c = self.convlstm(feature, lstm_state)

        # Policy branch
        policy_att = torch.sigmoid(self.policy_att(h))  # Mask-Attention
        policy_feature = self.policy_conv(h) * policy_att  # マスク処理
        policy_feature = policy_feature.view(policy_feature.size(0), -1)
        logit = self.policy_fc(policy_feature)

        # Value branch
        value_att = torch.sigmoid(self.value_att(h))  # Mask-Attention
        value_feature = self.value_conv(h) * value_att  # マスク処理
        value_feature = value_feature.view(value_feature.size(0), -1)
        value = self.value_fc(value_feature)

        return value, logit, (h, c), policy_att, value_att

    def init_state(self, batch_size, device):
        return self.convlstm.init_hidden(batch_size, self.conv_hw, device)

## エージェントの定義
エージェントが環境に対して確率的に行動し，`num_steps`ステップ分の状態価値・対数確率・報酬・エントロピーをそれぞれ収集します．基本的な流れは`actor_critic.ipynb`と同様ですが，ConvLSTMの隠れ状態（`lstm_state`）をエピソードが終了するまでステップをまたいで保持し続ける点が異なります．

rollout（`num_steps`分のステップ）をまたいだ誤差逆伝播（Truncated BPTT）を避けるため，隠れ状態はrolloutの終わりで`detach`します（学習ループ内で行います）．エピソードが終了した場合は，隠れ状態をゼロで初期化し直します．

In [ ]:
class Agent:
    def __init__(self, model, env, device):
        self.model = model
        self.env = env
        self.device = device
        self.state, _ = env.reset()
        self.done = False
        self.lstm_state = model.init_state(1, device)
        self.clear_actions()

    def action_train(self):
        state_v = torch.as_tensor(self.state).unsqueeze(0).to(self.device)
        value, logit, self.lstm_state, _, _ = self.model(state_v, self.lstm_state)

        prob = F.softmax(logit, dim=1)
        log_prob = F.log_softmax(logit, dim=1)
        entropy = -(log_prob * prob).sum(1)

        action = prob.multinomial(1).detach()
        log_prob_a = log_prob.gather(1, action)

        next_state, reward, terminated, truncated, info = self.env.step(action.item())
        self.done = terminated or truncated
        reward = max(min(reward, 1), -1)  # Reward clipping

        self.values.append(value)
        self.log_probs.append(log_prob_a)
        self.rewards.append(reward)
        self.entropies.append(entropy)

        self.state = next_state
        if self.done:
            self.state, _ = self.env.reset()
            self.lstm_state = self.model.init_state(1, self.device)

        return reward

    def clear_actions(self):
        self.values = []
        self.log_probs = []
        self.rewards = []
        self.entropies = []

## Lossの計算
Mask ACのLossは`actor_critic.ipynb`のActor-Criticと同様です．Valueでは最適状態価値を出力するように学習を行います．Loss計算は，次状態の推定の価値と実際に起こした行動から得られる価値の差を0にするように状態価値関数$V$を更新していくTD誤差を用いて更新します．

$$
L_{v}=(r+{\gamma}V(s_{t+1})-V(s_t))^2
$$

Policyでは，Valueの出力を利用したAdvantage $A(s)=r+{\gamma}V(s_{t+1})-V(s_t)$を用いて，最適な行動の確率を上げるように学習を行います．ここで，$H$はエントロピーであり，$\beta$はエントロピーの正則化項です．

$$
L_p=-\log(\pi(a|s))A(s)-\beta H(\pi)
$$

Lossの計算を行う関数を定義します．`calc_loss`関数ではPolicyとValueの両方のLossを計算します．

In [ ]:
def calc_loss(agent, R, gamma, entropy_coef, device):
    policy_loss = 0
    value_loss = 0

    for i in reversed(range(len(agent.rewards))):
        R = gamma * R + agent.rewards[i]
        advantage = R - agent.values[i]
        value_loss = value_loss + 0.5 * advantage.pow(2)

        policy_ad = advantage.detach()
        policy_loss = policy_loss - agent.log_probs[i] * policy_ad - entropy_coef * agent.entropies[i]

    return policy_loss, value_loss

## 学習
Mask ACを用いて学習を行います．学習環境はAtari環境のPongゲーム環境を用います．
`actor_critic.ipynb`と同様，`num_steps`ステップ分のrolloutを集めるたびにネットワークを更新するn-step Actor-Criticとして学習しますが，ConvLSTMの隠れ状態はrolloutをまたいでエピソード終了まで保持し続けます．

Pongは報酬が疎な環境であり，人間並みのスコアに到達するには非常に多くのフレーム数（数百万フレーム以上）が必要です．ここでは，学習の仕組みと損失・報酬の推移を確認できる範囲のフレーム数で実行します．

In [ ]:
GAMMA = 0.99
LEARNING_RATE = 1e-4
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5
num_steps = 20        # rolloutのステップ数
num_frame = 200000    # 収束にはこの数十倍以上のフレーム数が必要

acnet = MaskAC(env.observation_space.shape, env.action_space.n).to(device)
optimizer = optim.Adam(acnet.parameters(), lr=LEARNING_RATE)
agent = Agent(acnet, env, device)

frame_idx = 0
episode_reward = 0.0
total_rewards = []
record_reward = []
record_step = []

ts = time.time()
while frame_idx < num_frame:
    agent.clear_actions()
    for step in range(num_steps):
        reward = agent.action_train()
        episode_reward += reward
        frame_idx += 1
        if agent.done:
            total_rewards.append(episode_reward)
            episode_reward = 0.0
            mean_reward = np.mean(total_rewards[-20:])
            record_reward.append(mean_reward)
            record_step.append(frame_idx)
            if len(total_rewards) % 5 == 0:
                print('Frame {0}/{1}: episode {2}, mean reward {3:.3f}, time {4}'.format(
                    frame_idx, num_frame, len(total_rewards), mean_reward, datetime.timedelta(seconds=time.time() - ts)))
            break

    R = torch.zeros(1, 1, device=device)
    if not agent.done:
        with torch.no_grad():
            state_v = torch.as_tensor(agent.state).unsqueeze(0).to(device)
            value, _, _, _, _ = acnet(state_v, agent.lstm_state)
        R = value

    policy_loss, value_loss = calc_loss(agent, R, GAMMA, ENTROPY_COEF, device)

    optimizer.zero_grad()
    (policy_loss + VALUE_COEF * value_loss).backward()
    optimizer.step()

    # rolloutの境界でConvLSTMの隠れ状態をdetach（Truncated BPTT）
    agent.lstm_state = (agent.lstm_state[0].detach(), agent.lstm_state[1].detach())

## 学習時の平均スコアの推移
横軸フレーム数，縦軸平均スコアとしたグラフを描画してみます．

In [ ]:
fig = plt.figure()
plt.plot(record_step, record_reward, color="red")
plt.grid()
plt.xlabel("step")
plt.ylabel("mean reward")
plt.savefig("./mask_attention_step_per_reward.png")
plt.show()

## 評価
学習したネットワーク（エージェント）の行動とエージェントの注視領域（Mask-Attention）を確認してみます．`mask_attention`変数で，Policy branchとValue branchのどちらのAttentionを可視化するか選択できます．

Mask-Attentionは，ConvLSTMの出力（7×7）に対して計算されるため，元のゲーム画面（210×160）のサイズまで拡大し，ヒートマップとして元画面に重ね合わせて表示します．

In [ ]:
mask_attention = "Policy"  # "Policy" または "Value"

gym.register_envs(ale_py)
raw_env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False, render_mode='rgb_array')
eval_env = MaxAndSkipEnv(raw_env)
eval_env = FireResetEnv(eval_env)
eval_env = ProcessFrame84(eval_env)
eval_env = ImageToPyTorch(eval_env)
eval_env = ScaledFloatFrame(eval_env)

state, info = eval_env.reset()
lstm_state = acnet.init_state(1, device)
done = False
frames = []

acnet.eval()
with torch.no_grad():
    while not done:
        raw_frame = eval_env.render()  # 元のPong画面 (210, 160, 3)

        state_v = torch.as_tensor(state).unsqueeze(0).to(device)
        value, logit, lstm_state, policy_att, value_att = acnet(state_v, lstm_state)
        action = torch.argmax(logit, dim=1).item()

        att = policy_att if mask_attention == "Policy" else value_att
        att_map = att[0, 0].cpu().numpy()
        att_map = cv2.resize(att_map, (raw_frame.shape[1], raw_frame.shape[0]))
        att_map = (att_map * 255).astype(np.uint8)
        heatmap = cv2.applyColorMap(att_map, cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(raw_frame, 0.6, heatmap, 0.4, 0)
        frames.append(overlay)

        state, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated

eval_env.close()
print('frames:', len(frames))
frames_to_html5(frames, './mask_attention_{}.mp4'.format(mask_attention.lower()))

## 課題

1. `mask_attention`を`"Value"`に変更し，Policy branchとValue branchでAttentionの様子がどのように異なるか比較してみましょう．
2. `num_steps`（rolloutのステップ数）や`ENTROPY_COEF`（エントロピー正則化の重み）を変えて，学習の様子がどのように変わるか確認してみましょう．
3. Pong以外のゲームで学習してみましょう．
    * `gym.make()`での環境の指定を変更することで，任意の環境で学習できます．
    * 指定できる環境は，`ALE/Breakout-v5`や`ALE/MsPacman-v5`などがあります．詳しくは[ドキュメント](https://ale.farama.org/environments/)をチェックしてください．